In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

In [15]:
FOLDER = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara'
CARGA_FOLDER = "Insumos IMEPLAN/Transporte de Carga/Base de Datos"
MATRICES_DEMANDA_FOLDER = "Matriz Origen Destino/Demanda x PF"
OLD_ZONES_FOLDER = "red_shapefiles/zones_agebs_visum/con accesos carreteros"
NEW_ZONES_FOLDER = "Visum Objects/Zones/zonas finales"

In [28]:
columnas = ["Origen_ageb", "Destino_ageb"]


light_freight = pd.read_csv(os.path.join(FOLDER, CARGA_FOLDER, "matriz_2023_LIGHT_FREIGHT_por_hora_id_visum.csv"))
light_freight = light_freight[["Segmento","Hora","AGEB_Origen", "AGEB_Destino", "Id_zona_visum_origen", "Id_zona_visum_destino", "Total_Vehiculos_2023"]]
light_freight = light_freight.rename(columns={
    "Segmento": "Modo",
    "AGEB_Origen": "Origen",
    "AGEB_Destino": "Destino",
    "Id_zona_visum_origen": "Origen_ageb",
    "Id_zona_visum_destino": "Destino_ageb",
    "Total_Vehiculos_2023": "Ponderador"
})
light_freight = light_freight.dropna(subset=columnas).copy()
light_freight[columnas] = light_freight[columnas].astype(int)

heavy_freight = pd.read_csv(os.path.join(FOLDER, CARGA_FOLDER, "matriz_2023_HEAVY_FREIGHT_por_hora_id_visum.csv"))
heavy_freight = heavy_freight[["Segmento","Hora","AGEB_Origen", "AGEB_Destino", "Id_zona_visum_origen", "Id_zona_visum_destino", "Total_Vehiculos_2023"]]
heavy_freight = heavy_freight.rename(columns={
    "Segmento": "Modo",
    "AGEB_Origen": "Origen",
    "AGEB_Destino": "Destino",
    "Id_zona_visum_origen": "Origen_ageb",
    "Id_zona_visum_destino": "Destino_ageb",
    "Total_Vehiculos_2023": "Ponderador"
})
heavy_freight = heavy_freight.dropna(subset=columnas).copy()
heavy_freight[columnas] = heavy_freight[columnas].astype(int)

In [29]:
heavy_freight.head()

,Modo,Hora,Origen,Destino,Origen_ageb,Destino_ageb,Ponderador
0,HEAVY_FREIGHT,06:00-06:59,141200265,EXT_CHAPALA,8005,10005,16.825
1,HEAVY_FREIGHT,06:00-06:59,EXT_ZAPOTLANEJO_CUOTA,EXT_ZAPOTLANEJO_CUOTA,10004,10004,15.436
2,HEAVY_FREIGHT,06:00-06:59,140392569,EXT_ZAPOTLANEJO_CUOTA,210,10004,11.241
3,HEAVY_FREIGHT,06:00-06:59,EXT_ZAPOTLANEJO_CUOTA,140395366,10004,412,9.610
4,HEAVY_FREIGHT,06:00-06:59,EXT_COLIMA,EXT_ZAPOTLANEJO_CUOTA,10006,10004,9.268


In [30]:
old_zones = gpd.read_file(os.path.join(FOLDER, OLD_ZONES_FOLDER, "new_zones_with_access.shp"))
old_zones = old_zones[["clave_ageb", "id_visum"]]
old_zones = old_zones.rename(columns={"id_visum":"old_visum_id"})

new_zones = gpd.read_file(os.path.join(FOLDER, NEW_ZONES_FOLDER, "zonas_2212.shp"))
new_zones = new_zones.rename(columns={"id_visum":"new_visum_id"})

old_zones = old_zones.merge(new_zones[["clave_ageb", "new_visum_id"]], on="clave_ageb", how="left")
old_zones.head()

,clave_ageb,old_visum_id,new_visum_id
0,999990001,10001,10001
1,999990002,10002,10002
2,999990003,10003,10003
3,999990004,10004,10004
4,999990005,10005,10005


In [31]:
equivalencias = dict(zip(old_zones["old_visum_id"], old_zones["new_visum_id"]))

heavy_freight["Origen_ageb"] = heavy_freight["Origen_ageb"].map(equivalencias)
heavy_freight["Destino_ageb"] = heavy_freight["Destino_ageb"].map(equivalencias)
light_freight["Origen_ageb"] = light_freight["Origen_ageb"].map(equivalencias)
light_freight["Destino_ageb"] = light_freight["Destino_ageb"].map(equivalencias)

### Separar en periodos
6 matrices para light freight
6 matrices para heavy freight

| Period Code | Period Name | Start Hour | End Hour |
|--------------|----------|-------|-------|
| EM | Early Morning    | 00:00 | 03:59 |
| AP | AM Peak          | 04:00 | 08:59 |
| LM | Late Morning     | 9:00  | 11:59 |
| EA | Early Afternoon  | 12:00 | 17:59 |
| PP | PM Peak          | 18:00 | 20:59 |
| LN | Late Night       | 21:00 | 23:59 |


In [32]:
peak_factors = pd.read_csv(os.path.join(FOLDER, MATRICES_DEMANDA_FOLDER, "diccionario_peak_factors_v2.csv"))
pf_dict = (
    peak_factors
    .set_index("Periodo")["Peak factor por periodo"]
    .to_dict()
)
pf_dict

{'EM': 0.2789019162884518,
 'AP': 0.3676588103345247,
 'LM': 0.3429832296638419,
 'EA': 0.1938100575979761,
 'PP': 0.444205666866585,
 'LN': 0.4563756712876233}

In [33]:
periodos = ["EM", "AP", "LM", "EA", "PP", "LN"]

def asignar_periodo(df):
    df = df.copy()

    # Extrae la hora inicial:
    # "06:00-\n06:59" -> 6
    df["hora_inicio"] = (
        df["Hora"]
        .astype("string")
        .str.extract(r"(\d{1,2}):")[0]
        .astype(int)
    )

    df["Periodo"] = pd.cut(
        df["hora_inicio"],
        bins=[-1, 3, 8, 11, 17, 20, 23],
        labels=periodos
    )

    return df

In [34]:
light_freight = asignar_periodo(light_freight)
heavy_freight = asignar_periodo(heavy_freight)

# GET MATRICES BY PERIOD
light_matrices = {
    periodo: light_freight[
        light_freight["Periodo"] == periodo
    ].copy()
    for periodo in periodos
}

heavy_matrices = {
    periodo: heavy_freight[
        heavy_freight["Periodo"] == periodo
    ].copy()
    for periodo in periodos
}

In [35]:
light_matrices["AP"].dtypes

Modo                 str
Hora                 str
Origen               str
Destino              str
Origen_ageb        int64
Destino_ageb       int64
Ponderador       float64
hora_inicio        int64
Periodo         category
dtype: object

In [36]:
OUT_FOLDER = os.path.join(FOLDER, MATRICES_DEMANDA_FOLDER)

for tipo, matrices in [("LIGHT", light_matrices), ("HEAVY", heavy_matrices)]:
    for periodo, df_periodo in matrices.items():
        # Add PF column
        df_periodo["Peak Factor"] = df_periodo["Periodo"].map(pf_dict).astype(float)
        # Add viajes x PF column
        df_periodo["Ponderador_x_PF"] = df_periodo["Ponderador"] * df_periodo["Peak Factor"]

        # Save to CSV
        output_file = os.path.join(OUT_FOLDER, f"matriz_{tipo}_FREIGHT_{periodo}.csv")
        df_periodo.to_csv(output_file, index=False)

        print(f"Saved {output_file} with {len(df_periodo)} rows.")


Saved /Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Matriz Origen Destino/Demanda x PF/matriz_LIGHT_FREIGHT_EM.csv with 0 rows.
Saved /Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Matriz Origen Destino/Demanda x PF/matriz_LIGHT_FREIGHT_AP.csv with 418 rows.
Saved /Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Matriz Origen Destino/Demanda x PF/matriz_LIGHT_FREIGHT_LM.csv with 985 rows.
Saved /Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Matriz Origen Destino/Demanda x PF/matriz_LIGHT_FREIGHT_EA.csv with 1642 rows.
Saved /Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecno